# Representation and tabular controls

Install `.[tabular,notebooks]`. Compare CKA under transformations, train a concept separator on reference data, inspect held-out directional sensitivity, and compute PDP/ICE, permutation importance and constrained candidate counterfactuals. These are observational/perturbation diagnostics, not causal or clinical guarantees.

Run all cells in a fresh kernel. The final cell records local runtime and kernel RSS.

In [1]:
import os, time, json, resource, sys
os.environ["CUDA_VISIBLE_DEVICES"] = ""
os.environ["USE_TF"] = "0"
os.environ["WANDB_MODE"] = "disabled"
import torch, numpy as np
torch.set_num_threads(2)
torch.set_num_interop_threads(2)
torch.manual_seed(7)
np.random.seed(7)
started = time.perf_counter()
print("CPU, fixed seed 7; tiny synthetic fixtures; no weight downloads")


CPU, fixed seed 7; tiny synthetic fixtures; no weight downloads


In [2]:
from sklearn.tree import DecisionTreeClassifier
from autoexplain.representations import linear_cka, ridge_probe, concept_sensitivity
from autoexplain.concepts import fit_cav,nmf_features
from autoexplain.tabular_tools import permutation_importance,partial_dependence_ice,nearest_counterfactual
x=torch.randn(96,4); y=(x[:,0]>0).long()
print("CKA identity/scale:",linear_cka(x,x),linear_cka(x,3*x+2))
probe,pred=ridge_probe(x[:64],y[:64],x[64:])
print("Held-out probe accuracy:",float((pred==y[64:].numpy()).mean()))
cav=fit_cav(x[:64][y[:64]==1],x[:64][y[:64]==0])
gradient=torch.tensor([[1.,0.,0.,0.]]).expand(32,-1)
print("Directional sensitivity:",concept_sensitivity(gradient,cav["direction"]))
print("NMF reconstruction error:",nmf_features(x.abs(),components=2)["reconstruction_error"])
tree=DecisionTreeClassifier(max_depth=2,random_state=7).fit(x[:64].numpy(),y[:64].numpy())
print("Held-out permutation importance:",permutation_importance(tree,x[64:],y[64:],repeats=3).importances_mean)
pdp=partial_dependence_ice(tree,x[64:].numpy(),[0],grid_resolution=8)
print("ICE shape:",pdp.individual.shape)
cf=nearest_counterfactual(tree,x[64].numpy(),x[65:].numpy(),target=1-int(y[64]))
print("Counterfactual:",cf)


CKA identity/scale: 1.0 0.9999999999999988
Held-out probe accuracy: 0.96875
Directional sensitivity: 1.0
NMF reconstruction error: 7.475287221710754
Held-out permutation importance: [0.35416667 0.         0.         0.        ]
ICE shape: (1, 32, 8)
Counterfactual: {'counterfactual': array([-0.36739948, -1.51251018,  0.97445631,  0.53983772]), 'scaled_l1': 2.0655929115600884, 'target': 0, 'feasible_candidates': 10}


In [3]:
elapsed = time.perf_counter() - started
rss = resource.getrusage(resource.RUSAGE_SELF).ru_maxrss
rss_mib = rss / (1024**2 if sys.platform == "darwin" else 1024)
print(json.dumps({"tutorial_compute_wall_seconds": round(elapsed, 3),
                 "kernel_lifetime_peak_rss_mib": round(rss_mib, 2),
                 "memory_scope": "kernel only, including imports; not aggregate system memory"}))


{"tutorial_compute_wall_seconds": 0.58, "kernel_lifetime_peak_rss_mib": 409.74, "memory_scope": "kernel only, including imports; not aggregate system memory"}
